In [10]:
import pandas as pd

# 1. Load the FHFA All-Transactions file
data_path = '../data/raw/hpi_at_metro.csv'
df_raw = pd.read_csv(data_path, header=None)
df_raw.columns = ['metro_name', 'msa_code', 'year', 'quarter', 'index_sa', 'index_nsa']

# 2. Filter for strictly Kansas-related metro areas using regex
ks_filter = df_raw['metro_name'].str.contains(r', KS$|-KS', case=False, regex=True, na=False)
unique_ks_metros = df_raw[ks_filter]['metro_name'].unique()

print("Unique Kansas Metropolitan Areas:")
for metro in sorted(unique_ks_metros):
    print(f"- {metro}")

Unique Kansas Metropolitan Areas:
- Joplin, MO-KS
- Kansas City, MO-KS
- Lawrence, KS
- Manhattan, KS
- St. Joseph, MO-KS
- Topeka, KS
- Wichita, KS


# 1. Finding the Relevant Metro Areas
FILL THIS OUT

In [12]:
import pandas as pd
import numpy as np

# 1. Load the FHFA All-Transactions file with no headers
data_path = '../data/raw/hpi_at_metro.csv'
df_raw = pd.read_csv(data_path, header=None)

# 2. Assign proper column names based on FHFA MSA layout
df_raw.columns = ['metro_name', 'msa_code', 'year', 'quarter', 'index_value', 'standard_error']

# 3. Define your target Kansas metropolitan areas
target_ks_metros = [
    'Kansas City, MO-KS', 
    'Wichita, KS', 
    'Lawrence, KS', 
    'Manhattan, KS', 
    'Topeka, KS'
]

# 4. Filter for your target regions
df_ks = df_raw[df_raw['metro_name'].isin(target_ks_metros)].copy()

# 5. Clean index values: replace '-' with NaN, then convert to numeric
df_ks['index_value'] = pd.to_numeric(df_ks['index_value'].replace('-', np.nan), errors='coerce')

# Drop rows with missing index values (pre-collection historical quarters)
df_ks = df_ks.dropna(subset=['index_value'])

# 6. Sort chronologically by metro, year, and quarter
df_ks = df_ks.sort_values(by=['metro_name', 'year', 'quarter']).reset_index(drop=True)

# 7. Calculate your target variable: Quarterly Percentage Growth
df_ks['quarterly_growth'] = df_ks.groupby('metro_name')['index_value'].pct_change() * 100

# 8. Inspect the resulting clean dataset
print(f"Total valid quarterly observations: {df_ks.shape[0]}")
display(df_ks.head(10))

Total valid quarterly observations: 917


,metro_name,msa_code,year,quarter,index_value,standard_error,quarterly_growth
0,"Kansas City, MO-KS",28140,1976,1,44.94,( 0.73),NaN
1,"Kansas City, MO-KS",28140,1976,2,45.25,( 0.54),0.689809
2,"Kansas City, MO-KS",28140,1976,3,46.30,( 0.48),2.320442
3,"Kansas City, MO-KS",28140,1976,4,46.19,( 0.50),-0.237581
4,"Kansas City, MO-KS",28140,1977,1,47.15,( 0.57),2.078372
5,"Kansas City, MO-KS",28140,1977,2,49.72,( 0.45),5.450689
6,"Kansas City, MO-KS",28140,1977,3,51.61,( 0.40),3.801287
7,"Kansas City, MO-KS",28140,1977,4,53.88,( 0.49),4.398372
8,"Kansas City, MO-KS",28140,1978,1,55.66,( 0.55),3.303638
9,"Kansas City, MO-KS",28140,1978,2,58.04,( 0.48),4.275961


# 2. Finding Quarterly growth
FILL THIS OUT

In [13]:
import os

# Create processed folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Drop the initial NaN growth rows per metro so your modeling set is pristine
df_modeling_ready = df_ks.dropna(subset=['quarterly_growth']).copy()

# Save to your processed data folder
df_modeling_ready.to_csv('../data/processed/kansas_hpi_processed.csv', index=False)

print(f"Cleaned modeling dataset saved! Total rows ready for features: {df_modeling_ready.shape[0]}")
display(df_modeling_ready.head())

Cleaned modeling dataset saved! Total rows ready for features: 912


,metro_name,msa_code,year,quarter,index_value,standard_error,quarterly_growth
1,"Kansas City, MO-KS",28140,1976,2,45.25,( 0.54),0.689809
2,"Kansas City, MO-KS",28140,1976,3,46.30,( 0.48),2.320442
3,"Kansas City, MO-KS",28140,1976,4,46.19,( 0.50),-0.237581
4,"Kansas City, MO-KS",28140,1977,1,47.15,( 0.57),2.078372
5,"Kansas City, MO-KS",28140,1977,2,49.72,( 0.45),5.450689
